# 02 · Klasik modellerle saldırı tespiti (ikili: Normal / Saldırı)

Bu notebook projenin **Aşama 2**'sini adım adım anlatır. Amaç, aynı test kayıtlarında şu yöntemleri **adil** biçimde karşılaştırmaktır:

| Yöntem | Eğitimde ne görür? | Neden var? |
|---|---|---|
| **DummyClassifier** | Yalnızca eğitimdeki çoğunluk sınıfı | Alt sınır. Her model bundan iyi olmalı. |
| **Logistic Regression** | Tüm eğitim kayıtları + etiketleri | Basit, hızlı, doğrusal referans |
| **Random Forest** | Tüm eğitim kayıtları + etiketleri | Güçlü ağaç topluluğu |
| **XGBoost** | Tüm eğitim kayıtları + etiketleri | Gradyan artırmalı ağaçlar (kurulu değilse "atlandı" diye kaydedilir) |
| **Isolation Forest** | **Yalnızca Normal** eğitim kayıtları, etiketsiz | Saldırı örneği görmeden "olağandışı"yı bulmaya çalışır (anomali tespiti) |

**Altın kural:** Model ayarları (hiperparametreler) ve karar eşiği yalnızca **doğrulama (val)** verisiyle seçilir. **Test** verisi en sona, bir kez açılır.
Bu notebook'taki canlı örnekler testi hiç açmaz; test sonuçları `scripts/03_train_eval.py --evaluate-test` ile üretilip buradan okunur.

**Bu notebook'u çalıştırmadan önce** (proje kök klasöründe):
```bash
python scripts/01_audit_data.py --data /yol/InSDN_DatasetCSV
python scripts/02_make_split.py --data /yol/InSDN_DatasetCSV
python scripts/03_train_eval.py --data /yol/InSDN_DatasetCSV --evaluate-test
python scripts/04_make_figures.py
```
Veri klasörünü `INSDN_DIR` ortam değişkeniyle ya da aşağıdaki ilk kod hücresinde verebilirsiniz.

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 0 · Hazırlık

Proje klasörünü bulur, `src/` içindeki kütüphanemizi (`sdnids`) içe aktarır ve yolları ayarlar.
Notebook'taki kod **kütüphaneyi çağırır**; asıl kurallar (sızıntı koruması, şema denetimi) `src/sdnids/` içindedir ve testlerle sınanır.
Böylece eğitimde ve canlı Mininet hattında **aynı kod** kullanılır.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

# Veri klasörünüzü buraya yazabilir ya da INSDN_DIR ortam değişkenini kullanabilirsiniz.
DATA_DIR = Path(os.environ.get("INSDN_DIR", ROOT / "data" / "InSDN_DatasetCSV"))
OUT = Path(os.environ.get("SDNIDS_OUT", ROOT / "outputs"))
FIG_DIR = Path(os.environ.get("SDNIDS_FIG_DIR", ROOT / "figures"))
DEMO_FEATURE_SET = os.environ.get("SDNIDS_DEMO_FEATURE_SET", "llm_subset")
ALLOW_SYNTHETIC = os.environ.get("SDNIDS_ALLOW_SYNTHETIC") == "1"  # yalnızca otomatik testler için
FIG_DIR.mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sdnids.config import load_config
from sdnids.data.dedupe import deduplicate_vectors
from sdnids.data.loader import load_insdn
from sdnids.data.synthetic import require_real_data
from sdnids.experiment import load_design_context, make_split_data
from sdnids.metrics import binary_metrics, confusion_matrix_df
from sdnids.modeling import LLM_SUBSET_FEATURES, fit_model, get_model_spec, resolve_feature_set
from sdnids.plots import plot_class_distribution, plot_confusion, plot_metric_ci, plot_pr_curve, plot_score_hist, plot_success_latency
from sdnids.reporting import collect_results, collect_status, per_class_table
from sdnids.schema import FeatureSchema

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
cfg = load_config(ROOT / "configs" / "experiment.json")
print("Proje klasörü :", ROOT)
print("Veri klasörü  :", DATA_DIR)
print("Çıktı klasörü :", OUT)

## 1 · Veriyi yükle ve tekrarları indirge

InSDN'de kayıtların yaklaşık yarısı **birebir aynı özellik vektörü**dür (DDoS'ta 121.942 satır, yalnızca 1.520 farklı kayıt!).
Tekrarları indirgemezsek aynı kayıt hem eğitime hem teste düşebilir; model "ezberlediği" kaydı testte görür ve başarı **olduğundan yüksek** çıkar.

`deduplicate_vectors`:
* aynı vektörleri tek kayıtta toplar (kaç kopya olduğu `n_copies` sütununda saklanır, **model girdisi değildir**),
* aynı vektör iki farklı etiketle görülüyorsa (çelişkili etiket) o kayıtları dışlar ve listeler.

In [ ]:
df_raw = load_insdn(DATA_DIR, dayfirst=cfg["data"]["dayfirst"])
if not ALLOW_SYNTHETIC:
    require_real_data(df_raw)  # sentetik veriyle sonuç üretilmesini engeller

schema = FeatureSchema.load(OUT / "audit" / "schema.json")
df, excluded, rep = deduplicate_vectors(df_raw, schema.names)
print(f"{rep['rows_in']:,} satır -> {rep['rows_unique_kept']:,} benzersiz kayıt")
print(f"{rep['rows_excluded_conflicting_label']} satır çelişkili etiket nedeniyle dışlandı")
print(f"Özellik sayısı (şema): {len(schema.names)}")

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 2 · Kaydedilmiş bölmeyi aç

Bölme (train/val/test) Aşama 1'de üretildi ve diske yazıldı. `load_design_context`:
* test kayıt kimliklerinin **SHA-256 özetini** kontrol eder; dosya değiştirilmişse hata verir (test kümesi "dondurulmuştur"),
* veri ile bölme dosyasının birebir aynı kayıtlardan oluştuğunu denetler.

İki bölme tasarımımız var:
* **within_run:** aynı yakalamalardan rastgele (grup bütünlüğünü koruyarak) %60/%20/%20. *İyimser* bir tahmin verir.
* **cross_session:** test, eğitimde hiç görülmemiş bir yakalama oturumundan (`metasploitable-2.csv`) ve Normal trafiğin zamansal olarak son %20'sinden oluşur. *Genelleme* ölçer.

In [ ]:
ctx = load_design_context(df, schema, OUT / "split_within_run")
split_of = ctx.assignment.reindex(df["record_id"].astype(str)).to_numpy()
dist = pd.crosstab(df["label"], split_of)[[c for c in ("train", "val", "test") if c in set(split_of)]]
dist["toplam"] = dist.sum(axis=1)
dist.sort_values("toplam", ascending=False)

In [ ]:
fig = plot_class_distribution(dist.drop(columns="toplam"), title="Sınıf dağılımı (within_run, tekrarlar indirgenmiş)",
                              path=FIG_DIR / "nb02_sinif_dagilimi_within_run.png")
plt.show()

**Nasıl okunur?** Eksen logaritmiktir: her çizgi bir öncekinin 10 katıdır. U2R (17 kayıt), BOTNET ve Web-Attack çok azdır;
test bölmesinde 30'dan az örneği olan sınıflar için **güvenilir ölçüm iddia etmiyoruz**.

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 3 · Özellik kümeleri: neden iki tane?

* **full:** 77 akış özelliğinin tamamı (kimlik alanları hariç: IP, port, zaman damgası, flow_id, dosya adı **asla** girmez).
* **llm_subset:** 20 özellik. İleride **LLM'e de aynı 20 özelliği** vereceğiz. Adil karşılaştırma için klasik modelleri hem bu kümeyle hem tüm özelliklerle çalıştırıyoruz.

20 özellik **alan bilgisiyle** seçildi (hacim, hız, paket boyu, zamanlama, TCP bayrakları). Veriden ya da test sonucundan seçilmedi.
Birimler CICFlowMeter geleneğine göre *varsayılmıştır* (süreler mikrosaniye); InSDN belgesi birimleri yazmıyor.

In [ ]:
units = {f.name: f.unit_guess for f in schema.features}
pd.DataFrame({
    "özellik": list(LLM_SUBSET_FEATURES),
    "şemada var mı": [f in units for f in LLM_SUBSET_FEATURES],
    "birim (tahmin, doğrulanmadı)": [units.get(f) for f in LLM_SUBSET_FEATURES],
})

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 4 · Canlı örnek: Logistic Regression'ı eğit, DOĞRULAMADA değerlendir

Burada küçük bir örnekle bütün mekanizmayı görüyoruz. **Test bölmesi bu hücrede hiç oluşturulmaz** (`make_split_data` yalnızca `train` ve `val` ister).

Adımlar:
1. Ön işleme (sonsuz → eksik, eğitim medyanıyla doldurma, `log1p`, standartlaştırma) **yalnızca eğitimde öğrenilir**.
2. Model eğitilir.
3. Eşik iki şekilde belirlenir: `p0.5` (olasılık ≥ 0,5) ve `val_f1` (doğrulamada F1'i en iyi yapan eşik).

In [ ]:
feats = resolve_feature_set(DEMO_FEATURE_SET, schema)
data = make_split_data(ctx, ("train", "val"), feats)   # test bilerek oluşturulmuyor
tr, va = data["train"], data["val"]
print(f"eğitim: {len(tr.y):,} kayıt ({tr.y.mean():.1%} saldırı) | doğrulama: {len(va.y):,} kayıt ({va.y.mean():.1%} saldırı)")

spec = get_model_spec("logreg")
demo = fit_model(spec, {"C": 1.0, "class_weight": None}, 42, tr.X, tr.y, va.X, va.y,
                 feature_set=DEMO_FEATURE_SET, schema_digest=schema.digest())
s_val, preds = demo.predict_all(va.X)
print("Seçilen eşikler:", {k: (round(v, 4) if v is not None else None) for k, v in demo.thresholds.items()})
show = ["accuracy", "attack_precision", "attack_recall", "attack_f1", "fpr", "fnr", "pr_auc"]
pd.DataFrame({p: binary_metrics(va.y, preds[p], s_val) for p in preds}).loc[show].round(4)

**Metrikler ne anlatıyor?**
* **attack_precision:** "saldırı" dediklerimizin ne kadarı gerçekten saldırı.
* **attack_recall:** gerçek saldırıların ne kadarını yakaladık.
* **FPR (yanlış alarm oranı):** normal trafiğin ne kadarına yanlışlıkla alarm verdik. Bir ağ yöneticisi için en can sıkıcı hata budur.
* **FNR (kaçırma oranı):** saldırıların ne kadarını kaçırdık (= 1 − recall).
* **PR-AUC:** eşikten bağımsız; tüm eşiklerde precision–recall dengesinin özeti.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
for ax, pol in zip(axes, preds):
    m = binary_metrics(va.y, preds[pol])
    plot_confusion(m, title=f"Doğrulama, eşik kuralı: {pol}", ax=ax)
fig.tight_layout()
plt.show()

fig = plot_pr_curve(va.y, s_val, demo.thresholds.get("val_f1"), title="Logistic Regression: doğrulama PR eğrisi",
                    path=FIG_DIR / "nb02_pr_egrisi_logreg_val.png")
plt.show()

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 5 · Isolation Forest neden farklı?

Isolation Forest **denetimsiz** bir yöntemdir: eğitimde yalnızca **Normal** trafiği görür, saldırı örneği ve etiketi görmez.
Her kayda bir **anomali skoru** verir: "bu kayıt, öğrendiğim normal trafiğe ne kadar benzemiyor?"

Bu yüzden diğer modellerle **aynı bilgiyle eğitilmiş değildir**; sonuçları yan yana koyarken bunu unutmamak gerekir. İki eşik kuralı raporlanır:
* `train_normal_q99`: eğitim Normal'inin %99'luk dilimi. **Hiç etiket kullanmaz**, gerçek denetimsiz senaryo budur.
* `val_f1`: doğrulamada F1'i en iyi yapan eşik. Doğrulama **etiketlerini** kullanır, yani yarı denetimli bir kolaylıktır.

In [ ]:
spec_if = get_model_spec("iforest")
iso = fit_model(spec_if, {"max_samples": 256}, 42, tr.X, tr.y, va.X, va.y, feature_set=DEMO_FEATURE_SET,
                schema_digest=schema.digest(), model_cfg={"n_estimators": 100})
s_if, p_if = iso.predict_all(va.X)
fig = plot_score_hist(s_if, va.y, {"train_normal_q99": iso.thresholds["train_normal_q99"], "val_f1": iso.thresholds["val_f1"]},
                      title="Isolation Forest: doğrulamada skor dağılımları", path=FIG_DIR / "nb02_iforest_skorlar_val.png")
plt.show()
pd.DataFrame({p: binary_metrics(va.y, p_if[p], s_if) for p in p_if}).loc[show].round(4)

**Nasıl okunur?** Saldırıların skoru Normal'in sağında (daha "anormal") olsaydı Isolation Forest iyi çalışırdı.
Eğrilerin ne kadar üst üste bindiğine ve etiketsiz eşiğin (kesikli çizgi) saldırıların ne kadarını geçtiğine bakın.
Saldırı akışları çoğunlukla kısa ve tekdüzedir; normal trafik ise çok çeşitlidir. Bu yüzden "olağandışı = saldırı" varsayımı burada zayıf kalabilir.

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 6 · Tam deney sonuçları (betikle üretildi)

`scripts/03_train_eval.py --evaluate-test` her model için şunları yaptı:
1. Hiperparametre ızgarasını **doğrulama PR-AUC'si** ile taradı (ilk tohumla).
2. Seçilen ayarla **5 tohumla** yeniden eğitti (rastgelelik içeren modeller: RF, XGBoost, Isolation Forest).
3. Her tohumun eşiklerini o tohumun **doğrulama** skorlarından hesapladı.
4. Ayarlar sabitlendikten **sonra** test kümesini skorladı ve erişimi `test_access_log.json` dosyasına yazdı.

Tabloda her model için **birincil eşik kuralı** var (modelin ilk bildirilen kuralı). Değerler 5 tohumun ortalamasıdır, `_std` tohumlar arası sapmadır.
Grafiklerdeki nokta ve güven aralığı ise **birincil tohuma (42)** aittir; bu yüzden tablodaki ortalamadan biraz farklı olabilir.

In [ ]:
models_root = OUT / "models"
test_tab = collect_results(models_root, "test")
if test_tab.empty:
    print("Test sonucu yok. Önce: python scripts/03_train_eval.py --data <veri> --evaluate-test")
cols = ["tasarım", "özellik_kümesi", "model", "eşik_kuralı", "tohum_sayısı", "accuracy", "attack_precision", "attack_recall",
        "attack_f1", "attack_f1_std", "fpr", "fnr", "macro_f1", "pr_auc"]
primary = test_tab[test_tab["birincil_kural"]] if not test_tab.empty else test_tab
primary[[c for c in cols if c in primary.columns]].sort_values(["tasarım", "özellik_kümesi", "model"]).round(4)

Aynı modelin diğer eşik kuralı da önceden bildirildiği için raporlanır (ör. Isolation Forest için `val_f1`). Bunlar ayrı satırlardır, en iyisi seçilip öne çıkarılmaz:

In [ ]:
secondary = test_tab[~test_tab["birincil_kural"]] if not test_tab.empty else test_tab
secondary[[c for c in cols if c in secondary.columns]].sort_values(["tasarım", "özellik_kümesi", "model"]).round(4)

In [ ]:
status = collect_status(models_root)
status[status["status"] != "ok"][["design", "feature_set", "model", "status", "reason"]] if "reason" in status else status

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 7 · Grafikler: confusion matrix, güven aralıkları, başarı–gecikme

* **Confusion matrix:** satırlar gerçek sınıf, sütunlar tahmin. Hücrede sayı ve gerçek sınıfa göre yüzde var.
* **Güven aralığı (%95 GA):** test kayıtları **aynı dosya + 5 dakikalık zaman bloğu** kümeleri halinde yeniden örneklenerek (küme bootstrap) hesaplandı.
  Zamanda yakın akışlar birbirine benzediği için kayıtları tek tek bağımsız saymak aralığı yapay olarak daraltırdı.
* **Başarı–gecikme:** sağ üst değil **sol üst** iyidir (yüksek başarı, düşük gecikme). Gecikme ekseni logaritmiktir.

**Dikkat: saldırı F1'i saldırı oranına bağlıdır.** cross_session testinde kayıtların yaklaşık %80'i saldırıdır; her şeye "saldırı" diyen
Dummy bile yüksek bir saldırı F1'i alır (ama FPR = 1, yani her normal akışa alarm verir!). Bu yüzden saldırı F1'ini her zaman
**FPR** ve **macro-F1** (iki sınıfı eşit tartar) ile birlikte okuyun. Başarı–gecikme grafiğinde macro-F1 kullanılır.

In [ ]:
for design in sorted(set(test_tab["tasarım"])) if not test_tab.empty else []:
    t = test_tab[test_tab["tasarım"] == design]
    p = FIG_DIR / f"nb02_f1_ga_{design}.png"
    plot_metric_ci(t, "attack_f1", title=f"Saldırı F1 (test): {design}", xlabel="saldırı F1; birincil tohum, çubuk: %95 GA", path=p)
    plt.show()
    p = FIG_DIR / f"nb02_macro_f1_ga_{design}.png"
    plot_metric_ci(t, "macro_f1", title=f"Macro-F1 (test): {design}", xlabel="macro-F1; birincil tohum, çubuk: %95 GA", path=p)
    plt.show()
    p = FIG_DIR / f"nb02_fpr_ga_{design}.png"
    plot_metric_ci(t, "fpr", title=f"Yanlış alarm oranı (test): {design}", xlabel="FPR; birincil tohum, çubuk: %95 GA",
                   path=p, log=True)
    plt.show()
    plot_success_latency(t, title=f"Başarı ve gecikme: {design}", path=FIG_DIR / f"nb02_basari_gecikme_{design}.png")
    plt.show()

In [ ]:
import json

for design in sorted(set(test_tab["tasarım"])) if not test_tab.empty else []:
    fs = "full" if "full" in set(test_tab["özellik_kümesi"]) else sorted(set(test_tab["özellik_kümesi"]))[0]
    rows = test_tab[(test_tab["tasarım"] == design) & (test_tab["özellik_kümesi"] == fs) & test_tab["birincil_kural"]]
    models = list(rows["model"])
    fig, axes = plt.subplots(1, len(models), figsize=(3.1 * len(models), 3.1))
    axes = np.atleast_1d(axes)
    for ax, m in zip(axes, models):
        res = json.loads((models_root / design / fs / m / "test_metrics.json").read_text(encoding="utf-8"))
        pol = rows.set_index("model").loc[m, "eşik_kuralı"]
        plot_confusion(res["per_seed"][str(res["primary_seed"])][pol], title=f"{m} ({pol})", ax=ax)
    fig.suptitle(f"Test confusion matrix: {design}, {fs}", x=0.01, ha="left")
    fig.tight_layout()
    plt.show()

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 8 · Sınıf bazında sonuçlar

İkili model yalnızca "Normal / Saldırı" der. Yine de her özgün sınıfta ne kadar başarılı olduğunu görebiliriz:
* **Normal** için oran = doğru "Normal" denenlerin payı (özgüllük = 1 − FPR),
* **saldırı sınıfları** için oran = "saldırı" diye yakalananların payı (recall).

`güvenilir = False` olan satırlar 30'dan az örneğe dayanır; bu sınıflar için kesin yorum yapılmaz.
Wilson aralığı kayıtları bağımsız varsayar, ilişkili akışlarda iyimserdir.

In [ ]:
for design in sorted(set(test_tab["tasarım"])) if not test_tab.empty else []:
    for m in ("rf", "logreg", "iforest"):
        try:
            fs = "llm_subset" if (models_root / design / "llm_subset" / m / "test_metrics.json").exists() else "full"
            t = per_class_table(models_root, design, fs, m)
        except FileNotFoundError:
            continue
        print(f"\n== {design} / {fs} / {m} ==")
        print(t[["sınıf", "n", "doğru_sayısı", "oran", "oran_adı", "wilson_alt", "wilson_üst", "güvenilir"]].round(4).to_string(index=False))

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_

## 9 · Nasıl yorumlanmalı? Sınırlılıklar

1. **within_run neredeyse her zaman çok yüksek çıkar.** Eğitim ve test aynı yakalamalardan geldiği için bu sonuç *iyimser*dir. Asıl soru **cross_session**'da başarının ne kadar korunduğudur (araştırma sorusu 5).
2. **Yakalama ortamı karışıklığı:** Normal ve saldırı trafiği farklı yakalamalardan geliyor. Model "saldırıyı" değil "hangi yakalamadan geldiğini" öğrenmiş olabilir. Mininet'teki canlı deney bu yüzden önemlidir.
3. **Sınıf kapsamı:** cross_session testinde BOTNET ve Web-Attack yok, U2R eğitimde hiç yok (testte 17 kayıt). Bu sınıflar için genelleme iddiası yapılamaz.
4. **Küçük farklar:** Güven aralıkları örtüşüyorsa, ondalığın 3. hanesindeki farkı "üstünlük" diye yorumlamayın.
5. **Gecikme ölçümü** yalnızca model hesabını kapsar (tek iş parçacığı, tek kayıt). Özellik çıkarma, serileştirme ve ağ gecikmesi dahil değildir; uçtan uca gecikme Mininet aşamasında ayrıca ölçülecek.
6. **Isolation Forest** diğerleriyle aynı bilgiyle eğitilmedi (yalnızca Normal). Karşılaştırmada bu farkı belirtin.
7. **XGBoost** bu sonuçlarda yoksa paketi kurulu olmadığı içindir (`status.json` dosyasında "atlandı"). Kurduktan sonra `03_train_eval.py --models xgboost --evaluate-test` ile eklenebilir.

> **Benim notum:** _(Bu hücreye kendi açıklamanızı yazın. Çift tıklayıp düzenleyebilirsiniz.)_